In [9]:
import pandas as pd
import os

# ===========================================================
# 1. 파일 로드 및 설정
# ===========================================================

# 파일 경로 (사용자 환경에 맞게 수정됨)
subway_path = '../content/24년 서울교통공사_역별 일별 시간대별 승하차인원.csv'
weather_path = '../content/AWS시간별관측정보(2024년1월-2024년12월).csv'
save_filename = 'merged_subway_weather_final.csv'

# 파일을 안전하게 읽는 함수 (한글 깨짐 방지)
def read_csv_safe(path):
    if not os.path.exists(path):
        print(f"[오류] 파일을 찾을 수 없습니다: {path}")
        return pd.DataFrame()
    
    try:
        # 대부분의 공공데이터는 cp949 인코딩 사용
        return pd.read_csv(path, encoding='cp949')
    except UnicodeDecodeError:
        print(f"[알림] cp949 인코딩 실패, utf-8로 재시도합니다: {path}")
        return pd.read_csv(path, encoding='utf-8')

print("데이터 로딩 중...")
df_subway = read_csv_safe(subway_path)
df_weather = read_csv_safe(weather_path)

if df_subway.empty or df_weather.empty:
    print("데이터 로드에 실패하여 프로세스를 종료합니다.")

else:
    print(f"지하철 데이터 로드 완료: {df_subway.shape}")
    print(f"날씨 데이터 로드 완료: {df_weather.shape}")

    # ===========================================================
    # 2. 지하철 데이터 전처리
    # ===========================================================
    
    # 컬럼명 공백 제거
    df_subway.columns = df_subway.columns.str.strip()

    # 시간대 컬럼 추출 ('06' 또는 '~' 또는 '24'가 포함된 컬럼)
    time_cols = [c for c in df_subway.columns if '06' in c or '~' in c or '24' in c]

    # Melt 수행 (가로 -> 세로 변환)
    id_vars = ['날짜', '호선', '역번호', '역명', '구분']
    valid_id_vars = [col for col in id_vars if col in df_subway.columns]
    
    df_subway_melted = df_subway.melt(
        id_vars=valid_id_vars, 
        value_vars=time_cols,
        var_name='시간대_raw', 
        value_name='승객수'
    )

    # 시간 포맷 정제 함수
    def clean_subway_hour(text):
        text = str(text).strip()
        if '06시 이전' in text: return 5
        if '24시 이후' in text: return 24
        return int(text.split('~')[0].strip()) # '06 ~ 07' -> 6

    df_subway_melted['시간'] = df_subway_melted['시간대_raw'].apply(clean_subway_hour)
    df_subway_melted['날짜'] = pd.to_datetime(df_subway_melted['날짜'])

    # 승객수 콤마(,) 제거 및 숫자 변환
    if df_subway_melted['승객수'].dtype == 'object':
        df_subway_melted['승객수'] = df_subway_melted['승객수'].str.replace(',', '').astype(float)

    # ===========================================================
    # 3. 날씨 데이터 전처리 (핵심 수정 포함)
    # ===========================================================
    
    df_weather.columns = df_weather.columns.str.strip()
    target_location = '화성시' # 원하는 지역 설정
    
    # 지역 필터링
    if '시군명' in df_weather.columns:
        df_weather['시군명'] = df_weather['시군명'].astype(str).str.strip()
        df_weather_selected = df_weather[df_weather['시군명'] == target_location].copy()
    else:
        print("[경고] '시군명' 컬럼이 없어 전체 데이터를 사용합니다.")
        df_weather_selected = df_weather.copy()

    if df_weather_selected.empty:
        print(f"[오류] '{target_location}' 데이터가 없습니다.")
    else:
        # 날짜 변환 (20240101 -> datetime)
        df_weather_selected['날짜'] = pd.to_datetime(
            df_weather_selected['관측일자'].astype(str).str.split('.').str[0], 
            format='%Y%m%d', errors='coerce'
        )

        # [핵심 수정] 시간 변환 로직 (자동 감지)
        # 0시, 1시... 가 아니라 0, 100, 200... 1200 형태인 경우 100으로 나눔
        if df_weather_selected['관측시간'].max() > 24:
            print(f"[알림] 날씨 데이터 시간이 100단위(예: 1200)입니다. 100으로 나누어 변환합니다.")
            df_weather_selected['시간'] = (df_weather_selected['관측시간'] / 100).astype(int)
        else:
            print(f"[알림] 날씨 데이터 시간이 시단위(예: 12)입니다. 그대로 사용합니다.")
            df_weather_selected['시간'] = df_weather_selected['관측시간'].astype(int)

        # 필요한 컬럼만 선택
        desired_cols = ['날짜', '시간', '기온', '시간누적강우량']
        available_cols = [c for c in desired_cols if c in df_weather_selected.columns]
        df_weather_final = df_weather_selected[available_cols]

        # ===========================================================
        # 4. 데이터 병합 및 저장
        # ===========================================================
        
        # 날짜와 시간을 기준으로 병합 (Left Join)
        merged_df = pd.merge(df_subway_melted, df_weather_final, on=['날짜', '시간'], how='left')

        # 강수량 결측치는 0으로 채움 (비 안 옴으로 간주)
        if '시간누적강우량' in merged_df.columns:
            merged_df['시간누적강우량'] = merged_df['시간누적강우량'].fillna(0)
            
            # 검증용 출력
            rain_count = len(merged_df[merged_df['시간누적강우량'] > 0])
            print(f"\n[검증] 병합된 데이터 중 비가 온(강수량 > 0) 데이터 수: {rain_count}개")
            if rain_count == 0:
                print("[주의] 여전히 강수량이 모두 0입니다. 날짜/시간 매칭을 다시 확인하세요.")
        
        # CSV 파일 저장
        merged_df.to_csv(save_filename, index=False, encoding='UTF-8')
        print(f"\n[성공] '{save_filename}' 파일로 저장이 완료되었습니다!")
        
        # 미리보기
        print("=== 데이터 미리보기 ===")
        try:
            display(merged_df.head())
        except NameError:
            print(merged_df.head())

데이터 로딩 중...
[알림] cp949 인코딩 실패, utf-8로 재시도합니다: ../content/24년 서울교통공사_역별 일별 시간대별 승하차인원.csv
[알림] cp949 인코딩 실패, utf-8로 재시도합니다: ../content/AWS시간별관측정보(2024년1월-2024년12월).csv
지하철 데이터 로드 완료: (200100, 26)
날씨 데이터 로드 완료: (1150477, 18)
[알림] 날씨 데이터 시간이 시단위(예: 12)입니다. 그대로 사용합니다.

[검증] 병합된 데이터 중 비가 온(강수량 > 0) 데이터 수: 1637530개

[성공] 'merged_subway_weather_final.csv' 파일로 저장이 완료되었습니다!
=== 데이터 미리보기 ===


,날짜,호선,역번호,역명,구분,시간대_raw,승객수,시간,기온,시간누적강우량
0,2024-01-01,1호선,150,서울역,승차,06시 이전,383,5,-1.4,0.0
1,2024-01-01,1호선,150,서울역,승차,06시 이전,383,5,0.1,0.0
2,2024-01-01,1호선,150,서울역,승차,06시 이전,383,5,0.9,0.0
3,2024-01-01,1호선,150,서울역,승차,06시 이전,383,5,-1.0,0.0
4,2024-01-01,1호선,150,서울역,승차,06시 이전,383,5,-1.4,0.0
